# Energy Price Forecasting — Process Documentation

**Author:** Nazim  
**Project:** Dutch Wholesale Electricity Price Forecasting  
**Context:** OpenRemote integration — predicting day-ahead energy prices (EUR/MWh) for the Netherlands

---

This notebook documents the full process of designing and implementing a machine learning pipeline for energy price forecasting. It covers:

1. Problem definition
2. Data collection and sources
3. Dataset compilation and feature engineering
4. Preprocessing pipeline
5. Model architectures considered
6. Training setup and design choices
7. Evaluation methodology
8. Deployment as a REST API

---
## 1. Problem Definition

The goal is to forecast **wholesale electricity prices** (EUR/MWh) in the Dutch market up to **48 hours ahead**, given the past **168 hours** (1 week) of historical data.

This is a **multi-step time series regression** problem:
- **Input:** a lookback window of 168 hourly observations across 13 features
- **Output:** a forecast horizon of 48 hourly price values

Accurate short-term price forecasting is valuable for energy trading, demand response, and smart grid management. Prices are influenced by generation supply (especially renewables), electricity load, and weather — all of which vary on hourly timescales.

---
## 2. Data Sources

Three publicly available data sources were combined to build the dataset.

### 2.1 ENTSO-E Transparency Platform
The European Network of Transmission System Operators for Electricity provides open access to pan-European grid data.

- **Energy prices** (`data/raw/entso-transparency-platform/prices/`): Hourly day-ahead electricity prices for the Netherlands. Coverage: 2015–2025.
- **Total load forecasts** (`data/raw/entso-transparency-platform/load/`): Day-ahead total electricity load forecasts. Coverage: 2014–2025.
- **Wind & solar generation forecasts** (`data/raw/entso-transparency-platform/generation/`): Day-ahead and actual generation for onshore wind and solar. Coverage: 2014–2025.

### 2.2 TenneT (Dutch Transmission System Operator)
TenneT publishes settlement prices for the Dutch electricity market.

- **Settlement prices** (`data/raw/tennet/`): Hourly imbalance settlement prices. Coverage: 2018–2025.

### 2.3 Weather Data
Electricity prices are closely correlated with weather, particularly for renewable generation and heating/cooling demand.

- **Weather observations** (`data/raw/weather/weather_2015_2024.csv`): Hourly weather variables for the Netherlands including temperature (2m), cloud cover, wind speed (10m), and shortwave radiation.

### 2.4 EMBER (European Market for Electricity)
- Monthly European wholesale electricity price data, used as a cross-reference for data quality checks.

---
**Why these sources?** Each source contributes a different causal signal. Prices are driven by supply (generation), demand (load), and weather. Combining all three gives the model the context it needs to learn these relationships.

---
## 3. Dataset Compilation

All raw sources were merged into a single compiled dataset: **`dataset_2022_2025.csv`**

The decision to focus on 2022–2025 was driven by data completeness — all sources have good coverage in this range, and this period includes significant price volatility (energy crisis 2022), making it a challenging and realistic benchmark.

### Final Feature Set

| Column | Description | Source |
|--------|-------------|--------|
| `price` | Day-ahead electricity price (EUR/MWh) — **target** | ENTSO-E |
| `gen_day_ahead` | Day-ahead wind+solar generation forecast (MW) | ENTSO-E |
| `gen_intraday` | Intraday generation update (MW) | ENTSO-E |
| `gen_actual` | Actual renewable generation (MW) | ENTSO-E |
| `load_actual` | Actual electricity consumption (MW) | ENTSO-E |
| `load_forecast` | Day-ahead load forecast (MW) | ENTSO-E |
| `temp_2m` | Air temperature at 2m (°C) | Weather |
| `cloud_cover` | Cloud cover (%) | Weather |
| `wind_speed_10m` | Wind speed at 10m (m/s) | Weather |
| `shortwave_radiation` | Solar irradiance (W/m²) | Weather |
| `hour` | Hour of day (0–23) — time cyclical feature | Derived |
| `day_of_week` | Day of week (0–6) — captures weekday/weekend patterns | Derived |
| `month` | Month of year (1–12) — captures seasonal patterns | Derived |

---
## 4. Preprocessing Pipeline

The preprocessing is implemented in `forecasting/data_utils.py`. The pipeline runs automatically at training time.

### Steps

**Step 1 — Load & parse timestamps**  
The CSV loader auto-detects the timestamp column by trying a list of common names (`time`, `datetime`, `MTU (UTC)`, etc.). The DataFrame is sorted chronologically and non-numeric columns are dropped.

In [ ]:
# From forecasting/data_utils.py — timestamp detection
_TIMESTAMP_CANDIDATES = [
    'MTU (UTC)', 'MTU (CET/CEST)',
    'time', 'datetime', 'timestamp',
    'date', 'Date', 'DateTime', 'Timestamp', 'index',
]

# The loader iterates through candidates and sets the first match as the index.
# If none match, it falls back to parsing the first column as datetime.

**Step 2 — Drop sparse columns**  
Any column with more than 50% missing values is dropped. This prevents low-quality features from degrading model performance.

In [ ]:
# From forecasting/data_utils.py — sparse column removal
thresh = int(len(df) * 0.5)
df = df.dropna(axis=1, thresh=thresh)

**Step 3 — Fill remaining NaNs**  
For feature columns, remaining gaps are filled with forward-fill then back-fill (appropriate for time series where the most recent value is a reasonable estimate). Rows where the **target price is missing** are dropped entirely — there is no way to impute the label.

In [ ]:
# From forecasting/data_utils.py — NaN handling
df[feature_cols] = df[feature_cols].ffill().bfill()
df = df.dropna(subset=['price'])  # target must be present

**Step 4 — Normalisation**  
Features are scaled using `StandardScaler` (zero mean, unit variance). Crucially, the **price column is scaled separately** from the other features. This is necessary because during inference, we need to invert the price scaling to obtain predictions in EUR/MWh, without needing to invert the full feature scaler.

In [ ]:
# From forecasting/data_utils.py — separate scalers
from sklearn.preprocessing import StandardScaler

price_scaler = StandardScaler()
price_vals = price_scaler.fit_transform(df[['price']].values)

feat_scaler = StandardScaler()
feat_vals = feat_scaler.fit_transform(df[feature_cols].values)

# Combined input: [scaled_price | scaled_features]
import numpy as np
combined = np.hstack([price_vals, feat_vals])

**Step 5 — Sliding window construction**  
The time series is split into overlapping windows. Each sample consists of:
- **X**: the past `lookback=168` hours of all features
- **y**: the next `horizon=48` hours of price only

The 168-hour (1 week) lookback was chosen to capture weekly seasonality. The 48-hour horizon corresponds to the typical day-ahead market clearing window.

In [ ]:
# From forecasting/data_utils.py — sliding window
lookback = 168  # 1 week
horizon  = 48   # 2 days ahead

X, y = [], []
for i in range(lookback, len(combined) - horizon + 1):
    X.append(combined[i - lookback: i])      # shape: (168, n_features)
    y.append(price_vals[i: i + horizon, 0])  # shape: (48,)

X = np.array(X)  # (N, 168, n_features)
y = np.array(y)  # (N, 48)

**Step 6 — Train / val / test split**  
The data is split chronologically (no shuffling) to avoid data leakage from future to past:
- **75%** training
- **10%** validation (used for early stopping and learning rate scheduling)
- **15%** test (held out until final evaluation)

In [ ]:
# From forecasting/data_utils.py — chronological split
n = len(X)
test_size  = int(n * 0.15)
val_size   = int(n * 0.10)
train_size = n - val_size - test_size

X_train, y_train = X[:train_size], y[:train_size]
X_val,   y_val   = X[train_size: train_size + val_size], y[train_size: train_size + val_size]
X_test,  y_test  = X[train_size + val_size:], y[train_size + val_size:]

---
## 5. Model Architectures

Five model architectures were implemented to allow a fair comparison. All are implemented in `forecasting/models.py` as PyTorch `nn.Module` subclasses with a shared interface: they take input of shape `(batch, seq_len, n_features)` and output `(batch, horizon)`.

### 5.1 LSTM (Long Short-Term Memory)

A stacked LSTM encoder. The final hidden state is passed through a linear layer to produce the forecast. LSTMs are a natural baseline for time series — they were designed to capture long-range temporal dependencies via gated memory cells.

In [ ]:
# From forecasting/models.py
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, horizon, hidden_size=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):          # x: (B, T, input_size)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # use last time step -> (B, horizon)

### 5.2 GRU (Gated Recurrent Unit)

A simpler variant of the LSTM with fewer parameters. GRUs merge the forget and input gates into a single update gate, making them faster to train while often matching LSTM performance on medium-length sequences.

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, horizon, hidden_size=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

### 5.3 BiLSTM (Bidirectional LSTM)

Processes the sequence in both directions (past → future and future → past) and concatenates the results. The bidirectional pass doubles the effective hidden size. In a training context, this is valid since both directions are available over the lookback window — only the forecast output is causal.

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, horizon, hidden_size=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size * 2, horizon)  # *2 for both directions

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

### 5.4 Transformer

A vanilla Transformer encoder where **time steps are treated as tokens**. Each hour in the lookback window attends to all other hours via multi-head self-attention. Positional encodings (sinusoidal) are added to inject temporal ordering, which the attention mechanism is otherwise order-agnostic about.

The last token's representation is projected to the forecast horizon.

In [ ]:
import math
import torch

class _PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding — injects temporal order into the Transformer."""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerModel(nn.Module):
    def __init__(self, input_size, horizon, d_model=128, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)   # project features -> d_model
        self.pos_enc    = _PositionalEncoding(d_model, dropout)
        enc_layer       = nn.TransformerEncoderLayer(d_model, nhead,
                                                     dim_feedforward=d_model * 4,
                                                     dropout=dropout, batch_first=True)
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers)
        self.fc         = nn.Linear(d_model, horizon)

    def forward(self, x):          # x: (B, T, input_size)
        x = self.pos_enc(self.input_proj(x))   # (B, T, d_model)
        x = self.encoder(x)                     # (B, T, d_model)
        return self.fc(x[:, -1, :])             # last token -> (B, horizon)

### 5.5 iTransformer (Inverted Transformer)

The iTransformer is based on the paper *"iTransformer: Inverted Transformers Are Effective for Time Series Forecasting"* (Liu et al., ICLR 2024).

The key insight is to **invert the token axis**: instead of treating each time step as a token, each **variate (feature)** becomes a token. The embedding for each variate is its full time series projected into `d_model` dimensions. Self-attention then models **cross-variate dependencies** — e.g., how wind speed correlates with generation, and how that affects price.

This architecture was included because electricity prices have strong multivariate dependencies (generation, load, weather all interact), which the standard Transformer with time-step tokens is less suited to capture explicitly.

In [ ]:
class iTransformerModel(nn.Module):
    """
    Each variate's full time series is projected to a d_model embedding.
    Self-attention captures cross-variate dependencies.
    """
    def __init__(self, input_size, seq_len, horizon, d_model=128, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.variate_proj = nn.Linear(seq_len, d_model)   # (seq_len,) -> (d_model,) per variate
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model * 4,
                                               dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.fc = nn.Linear(d_model * input_size, horizon)

    def forward(self, x):          # x: (B, T, V)
        x = x.permute(0, 2, 1)     # -> (B, V, T)  — variates become tokens
        x = self.variate_proj(x)    # -> (B, V, d_model)
        x = self.encoder(x)         # -> (B, V, d_model)
        x = x.flatten(1)            # -> (B, V * d_model)
        return self.fc(x)           # -> (B, horizon)

### Architecture Comparison Summary

| Model | Tokens | What attention captures | Parameters |
|-------|--------|------------------------|------------|
| LSTM | — | Sequential temporal patterns (gated memory) | ~200K |
| GRU | — | Same as LSTM, simpler gating | ~150K |
| BiLSTM | — | Temporal patterns in both directions | ~400K |
| Transformer | Time steps | Temporal dependencies across the window | ~300K |
| iTransformer | Variates | Cross-variate dependencies | ~500K |

*Parameter counts are approximate for default settings (hidden=128, d_model=128, layers=2, 13 input features, horizon=48).*

---
## 6. Training Setup

The training loop is implemented in `forecasting/trainer.py`.

### Loss function
**Mean Squared Error (MSE)** on the scaled price values. MSE penalises large errors more strongly than MAE, which is appropriate for energy prices where large spikes (e.g., during the 2022 energy crisis) are especially important to predict correctly.

### Optimiser
**Adam** with learning rate `1e-3`. Adam is the standard choice for deep learning — it adapts per-parameter learning rates and handles sparse gradients well.

### Learning rate scheduling
**ReduceLROnPlateau** halves the learning rate when validation loss stops improving for 5 consecutive epochs. This allows the model to fine-tune once it converges to a local region.

### Early stopping
Training stops if the validation loss does not improve for `patience=10` epochs. The best model weights (lowest validation loss) are restored at the end. This prevents overfitting and reduces unnecessary compute.

In [ ]:
# From forecasting/trainer.py — training loop structure
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()

best_val_loss = float('inf')
no_improve    = 0
patience      = 10

for epoch in range(1, epochs + 1):
    # --- Train ---
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

    # --- Validate ---
    model.eval()
    # ... compute val_loss ...
    scheduler.step(val_loss)

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            break  # restore best weights outside loop

model.load_state_dict(best_state)

**Gradient clipping** (`max_norm=1.0`) is applied to prevent exploding gradients, which is especially important for deep RNNs on long sequences.

### Hardware
The pipeline auto-detects CUDA (`--device auto`). Training was performed on a GPU. For RTX 5000-series (Blackwell) GPUs, a nightly PyTorch build with CUDA 12.8 support is required.

---
## 7. Evaluation

All models are evaluated on the **held-out test set** (last 15% of data, chronologically). Predictions are inverse-transformed back to EUR/MWh before computing metrics.

Three metrics are reported, each capturing a different aspect of forecast quality:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | mean(\|pred − actual\|) | Average error in EUR/MWh |
| **RMSE** | sqrt(mean((pred − actual)²)) | Penalises large errors more; sensitive to price spikes |
| **MAPE** | mean(\|pred − actual\| / \|actual\|) × 100 | Relative error in %; near-zero actuals are excluded |

In [ ]:
# From forecasting/trainer.py — metrics
import numpy as np

def compute_metrics(preds, targets):
    mae  = float(np.mean(np.abs(preds - targets)))
    rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))

    # Exclude near-zero actuals to avoid division by zero / inf MAPE
    mask = np.abs(targets) > 1e-3
    mape = float(np.mean(np.abs((preds[mask] - targets[mask]) / targets[mask])) * 100)

    return {'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'MAPE': round(mape, 4)}

### Running the benchmark

All five models can be trained and evaluated in one command:

```bash
python benchmark.py --data dataset_2022_2025.csv
```

This produces:
- `results/metrics.csv` — MAE, RMSE, MAPE, parameter count, and training time per model
- `results/comparison_mae.png`, `comparison_rmse.png`, `comparison_mape.png` — bar charts
- `results/{model}_predictions.png` — actual vs. predicted on the first test window
- `results/all_models_comparison.png` — all models overlaid on the same test window

---
## 8. Deployment as a REST API

The trained model is served via a **FastAPI** inference server (`app.py`), containerised with Docker.

### Design
- The model is loaded once at startup using FastAPI's `lifespan` context manager — no per-request model loading overhead.
- The same two-scaler scheme from training is used at inference: price and features are scaled separately before being fed to the model, then the prediction is inverse-transformed.
- Input validation is strict: the server checks that the input shape exactly matches `(seq_len, n_features)` from the training config stored in the checkpoint.

### API Endpoints

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/health` | Returns model status, architecture name, horizon, and seq_len |
| `POST` | `/predict` | Accepts `features` (2D list of shape `[168, 13]`), returns 48 price forecasts in EUR/MWh |

In [ ]:
# Example predict request (conceptual)
import json

request_body = {
    "features": [
        # 168 rows x 13 columns — last week of hourly observations
        [65.2, 4200.0, 4150.0, 4100.0, 14500.0, 14600.0, 12.5, 40.0, 5.2, 120.0, 14, 2, 3],
        # ... 167 more rows
    ]
}

# Response:
response = {
    "predictions": [63.1, 64.5, 67.2, "..."],  # 48 hourly prices in EUR/MWh
    "horizon": 48,
    "model": "transformer"
}

print(json.dumps(response, indent=2))

### Docker

The service is containerised for reproducibility and deployment:

```bash
# Build and run
docker build -t ml-forecast .
docker run -p 8000:8000 -v $(pwd)/saved_models:/app/saved_models ml-forecast
```

Environment variables allow the model name, directory, host, and port to be configured without rebuilding the image.

---
## 9. Summary of Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Forecast horizon | 48 hours | Matches day-ahead market clearing cycle |
| Lookback window | 168 hours (1 week) | Captures weekly seasonality (weekday/weekend patterns) |
| Target variable | `price` only | Forecasting a single target keeps the output head simple |
| Separate price scaler | Yes | Required for correct inverse-transform at inference |
| Train/val/test split | 75/10/15 chronological | No data leakage; val set used only for early stopping |
| Loss function | MSE | Penalises large errors; important for price spike periods |
| Gradient clipping | `max_norm=1.0` | Stabilises training for deep RNNs |
| Five architectures | LSTM, GRU, BiLSTM, Transformer, iTransformer | Covers the spectrum from classical RNNs to state-of-the-art attention models |
| iTransformer included | Yes | Specifically designed for multivariate time series with cross-variate attention |
| Deployment | FastAPI + Docker | Lightweight, production-ready, integrates with OpenRemote via REST |